# 07 · Inverses and the pseudoinverse / Inversas y la pseudoinversa

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb)

*Part IV · exercise · 15 min*

Use one question throughout this notebook:

> **Does an exact inverse exist? If not, what useful answer does the pseudoinverse give us instead?**

> 🇪🇸 Usa una sola pregunta durante todo el cuaderno:
>
> **¿Existe una inversa exacta? Si no existe, ¿qué respuesta útil nos entrega la pseudoinversa?**

## What you will be able to do / Lo que podrás hacer

- Explain an ordinary inverse as an exact **undo operation**.
- Diagnose when a square matrix is singular using **rank**, not only an exception message.
- Understand the pseudoinverse as a practical replacement when an ordinary inverse is unavailable.
- Distinguish **wide**, **square**, and **tall** systems.
- Solve a real `20,433 × 7` California-housing least-squares problem.
- Unfold real digit images into a wide matrix and interpret the **minimum-norm** solution.

> 🇪🇸
>
> - Explicar una inversa ordinaria como una operación exacta de **deshacer**.
> - Diagnosticar cuándo una matriz cuadrada es singular usando el **rango**, no solamente un mensaje de error.
> - Entender la pseudoinversa como una alternativa práctica cuando la inversa ordinaria no está disponible.
> - Distinguir sistemas **anchos**, **cuadrados** y **altos**.
> - Resolver un problema real de mínimos cuadrados de vivienda en California con forma `20.433 × 7`.
> - Desplegar imágenes reales de dígitos en una matriz ancha e interpretar la solución de **norma mínima**.

## Start with an everyday analogy / Empecemos con una analogía cotidiana

### Ordinary inverse / Inversa ordinaria

Think of an inverse as a perfect **Undo** button.

If:

`A` transforms `x` into `b`

then:

`A⁻¹` takes `b` back to exactly one `x`.

But that perfect undo button requires a very special situation:

- the matrix must be **square**;
- it must contain enough independent information — **full rank**.

### Pseudoinverse / Pseudoinversa

The pseudoinverse `A⁺` is the useful fallback when the perfect undo button is unavailable.

Depending on the geometry, it answers a different practical question:

- **Tall / Alta:** “Which solution fits all these observations as closely as possible?”
- **Wide / Ancha:** “Among many exact solutions, which one is the smallest/simplest in Euclidean norm?”
- **Singular square / Cuadrada singular:** “The ordinary inverse does not exist; give me the well-defined pseudoinverse solution instead.”

> 🇪🇸 Piensa en la inversa como un botón perfecto de **Deshacer**.
>
> La pseudoinversa `A⁺` es la alternativa útil cuando ese botón perfecto no existe.
>
> - **Sistema alto:** busca el mejor ajuste por mínimos cuadrados.
> - **Sistema ancho:** elige, entre muchas soluciones exactas, la de menor norma.
> - **Matriz cuadrada singular:** la inversa ordinaria no existe, pero la pseudoinversa sí.

## Three matrix geometries / Tres geometrías de matrices

Suppose the matrix has shape:

`(number of equations, number of unknowns)`

| Geometry / Geometría | Example / Ejemplo | Plain meaning / Significado sencillo | Pseudoinverse role / Papel de `pinv` |
|---|---:|---|---|
| **Wide / Ancha** | `(5, 7)` | fewer equations than unknowns / menos ecuaciones que incógnitas | minimum-norm exact solution when possible / solución exacta de norma mínima cuando es posible |
| **Square / Cuadrada** | `(7, 7)` | same number / misma cantidad | ordinary inverse only if full rank / inversa ordinaria solo si tiene rango completo |
| **Tall / Alta** | `(20,433, 7)` | many more equations than unknowns / muchas más ecuaciones que incógnitas | least-squares solution / solución de mínimos cuadrados |

### One sentence to remember / Una frase para recordar

> **Shape tells us the geometry; rank tells us whether the available information is independent enough.**

> 🇪🇸
>
> **La forma indica la geometría; el rango indica si la información disponible es suficientemente independiente.**

## Setup / Preparación

This notebook uses two real datasets:

1. **California housing districts.** After removing 207 rows with a missing value, 20,433 observations remain.
2. **Handwritten digit images** from `sklearn.datasets.load_digits()`.

The six housing features are standardized before regression. We also add a bias/intercept column, giving seven model columns in total.

> 🇪🇸 Este cuaderno usa dos conjuntos de datos reales:
>
> 1. **Distritos de vivienda de California.** Después de eliminar 207 filas con un valor faltante quedan 20.433 observaciones.
> 2. **Imágenes de dígitos manuscritos** de `sklearn.datasets.load_digits()`.
>
> Las seis variables de vivienda se estandarizan antes de la regresión y agregamos una columna de intercepto, para un total de siete columnas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

HOUSING = (
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
    "datasets/housing/housing.csv"
)

housing = pd.read_csv(HOUSING).dropna().reset_index(drop=True)

features = [
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]

X_raw = housing[features].to_numpy(float)
feature_mean = X_raw.mean(axis=0)
feature_std = X_raw.std(axis=0)
X_scaled = (X_raw - feature_mean) / feature_std

# Bias + six standardized real features -> seven columns.
X = np.column_stack([np.ones(len(housing)), X_scaled])
y = housing["median_house_value"].to_numpy(float)
column_names = ["bias"] + features

digits = load_digits()
digit_tensor = digits.images.astype(float)  # (1797, 8, 8)

def unfold(T, axis=0):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print("Housing rows / Filas de vivienda:", len(housing))
print("Housing design matrix / Matriz de diseño:", X.shape)
print("Target / Objetivo:", y.shape)
print("Digit tensor / Tensor de dígitos:", digit_tensor.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## Why this matters / Por qué esto importa

In real machine learning, `A⁻¹` is often **not** the right tool.

A design matrix commonly has:

- thousands of observations;
- only a few model coefficients.

For California housing:

`X.shape = (20433, 7)`

That means:

**20,433 equations trying to estimate 7 coefficients.**

The matrix is rectangular, so an ordinary inverse is not even defined.

The pseudoinverse lets us solve the problem in the correct sense.

### Learning cycle / Ciclo de aprendizaje

Before every exercise:

**Predict → Run → Explain / Predice → Ejecuta → Explica**

Ask:

1. Is the matrix wide, square, or tall?
2. What is its rank?
3. Can an ordinary inverse exist?
4. What should `A⁺b` mean here?

> 🇪🇸 En aprendizaje automático real, `A⁻¹` muchas veces **no** es la herramienta adecuada.
>
> En vivienda de California, `X.shape = (20433,7)` significa **20.433 ecuaciones para estimar 7 coeficientes**.
>
> La matriz es rectangular, por lo que una inversa ordinaria no está definida. La pseudoinversa permite resolver el problema con el significado matemático correcto.

### Interactive geometry translator / Traductor interactivo de geometría

Choose the number of rows while keeping seven columns.

Watch the same seven unknown coefficients move through:

**wide → square → tall**

> 🇪🇸 Elige el número de filas manteniendo siete columnas.
>
> Observa cómo los mismos siete coeficientes pasan por:
>
> **ancho → cuadrado → alto**

In [ ]:
geometry_rows = widgets.IntSlider(
    value=7,
    min=3,
    max=20,
    step=1,
    description="Rows / Filas:",
    continuous_update=False,
    style={"description_width": "110px"},
)

def explain_geometry(n_rows):
    n_cols = X.shape[1]

    if n_rows < n_cols:
        geometry = "wide / ancha"
        en = "fewer equations than unknowns; exact solutions may be non-unique."
        es = "menos ecuaciones que incógnitas; las soluciones exactas pueden no ser únicas."
    elif n_rows == n_cols:
        geometry = "square / cuadrada"
        en = "same number of equations and unknowns; invertibility still depends on rank."
        es = "igual número de ecuaciones e incógnitas; la invertibilidad todavía depende del rango."
    else:
        geometry = "tall / alta"
        en = "more equations than unknowns; least squares is the usual interpretation."
        es = "más ecuaciones que incógnitas; mínimos cuadrados es la interpretación habitual."

    A = X[:n_rows]
    rank = np.linalg.matrix_rank(A)

    print("Shape / Forma:", A.shape)
    print("Geometry / Geometría:", geometry)
    print("Rank / Rango:", rank)
    print("EN:", en)
    print("ES:", es)

geometry_output = widgets.interactive_output(
    explain_geometry,
    {"n_rows": geometry_rows},
)

display(widgets.VBox([geometry_rows, geometry_output]))

## Exercise 1 — make a real matrix singular / Ejercicio 1 — convierte una matriz real en singular

We select seven real California districts and all seven design columns:

`A_real.shape = (7,7)`

Then we deliberately copy one feature column onto another:

`A_singular[:, 2] = A_singular[:, 1]`

Now two columns contain exactly the same information.

### What does “singular” mean? / ¿Qué significa “singular”?

A square matrix is singular when it does not have enough independent directions to be reversed uniquely.

If two columns are identical:

- one column adds no new information;
- rank drops below `7`;
- an ordinary mathematical inverse does not exist.

### Important numerical note / Nota numérica importante

Do **not** use “did `np.linalg.inv` raise an exception?” as the mathematical definition of singularity.

Because computers use floating-point arithmetic, an `inv` call on a theoretically rank-deficient matrix may sometimes raise `LinAlgError`, but in some numerical situations it may also return a wildly unstable array.

The reliable teaching diagnostic here is:

`rank < number of columns`

We also inspect the smallest singular value and condition number.

> 🇪🇸 No uses “¿`np.linalg.inv` produjo un error?” como definición matemática de singularidad.
>
> Debido a la aritmética de punto flotante, el comportamiento numérico puede variar.
>
> El diagnóstico matemático que usaremos es:
>
> **`rango < número de columnas`**
>
> También observaremos el valor singular más pequeño y el número de condición.

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Select seven spread-out real rows:
#       row_idx = np.linspace(0, len(X) - 1, 7, dtype=int)
#       A_real = X[row_idx]
# 2. Print its rank.
# 3. Copy A_real, then duplicate one feature column:
#       A_singular[:, 2] = A_singular[:, 1]
# 4. Print the new rank.
# 5. Decide mathematically whether an ordinary inverse exists.
# 6. Inspect singular values and the condition number.
# 7. Compute A_plus = np.linalg.pinv(A_singular).
# 8. Verify the four Moore–Penrose conditions.
#
# ES:
# 1. Selecciona siete filas reales distribuidas.
# 2. Imprime su rango.
# 3. Duplica deliberadamente una columna de características.
# 4. Imprime el nuevo rango.
# 5. Decide matemáticamente si existe una inversa ordinaria.
# 6. Inspecciona los valores singulares y el número de condición.
# 7. Calcula A_plus = np.linalg.pinv(A_singular).
# 8. Verifica las cuatro condiciones de Moore–Penrose.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

row_idx = np.linspace(0, len(X) - 1, 7, dtype=int)
A_real = X[row_idx].copy()

A_singular = A_real.copy()
A_singular[:, 2] = A_singular[:, 1]

rank_real = np.linalg.matrix_rank(A_real)
rank_singular = np.linalg.matrix_rank(A_singular)
is_singular = rank_singular < A_singular.shape[1]

singular_values = np.linalg.svd(A_singular, compute_uv=False)
condition = np.linalg.cond(A_singular)

print("Original / Original:", A_real.shape, "rank/rango =", rank_real)
print("Duplicated / Duplicada:", A_singular.shape, "rank/rango =", rank_singular)
print()
print("Mathematically singular? / ¿Singular matemáticamente?:", is_singular)
print("Smallest singular value / Menor valor singular:", f"{singular_values[-1]:.3e}")
print("Condition number / Número de condición:", f"{condition:.3e}")
print()

if is_singular:
    print("EN: rank < 7, so an ordinary mathematical inverse does NOT exist.")
    print("ES: rango < 7, por lo tanto NO existe una inversa matemática ordinaria.")
else:
    print("EN: this matrix is full rank.")
    print("ES: esta matriz tiene rango completo.")

print()
print("EN: an exception from np.linalg.inv is NOT our definition of singularity.")
print("ES: una excepción de np.linalg.inv NO es nuestra definición de singularidad.")

A_plus = np.linalg.pinv(A_singular)

mp_checks = {
    "A A+ A = A": np.allclose(
        A_singular @ A_plus @ A_singular,
        A_singular,
    ),
    "A+ A A+ = A+": np.allclose(
        A_plus @ A_singular @ A_plus,
        A_plus,
    ),
    "(A A+)T = A A+": np.allclose(
        (A_singular @ A_plus).T,
        A_singular @ A_plus,
    ),
    "(A+ A)T = A+ A": np.allclose(
        (A_plus @ A_singular).T,
        A_plus @ A_singular,
    ),
}

print()
print("Moore–Penrose checks / Comprobaciones Moore–Penrose:")
for name, passed in mp_checks.items():
    print(f"  {name}: {passed}")

### Interactive singularity explorer / Explorador interactivo de singularidad

Switch between:

- the original real `7×7` matrix;
- the version with a duplicated feature.

Watch:

- rank;
- smallest singular value;
- condition number;
- pseudoinverse reconstruction error.

> 🇪🇸 Cambia entre la matriz real original `7×7` y la versión con una característica duplicada.
>
> Observa el rango, el menor valor singular, el número de condición y el error de reconstrucción con la pseudoinversa.

In [ ]:
matrix_choice = widgets.ToggleButtons(
    options=[
        ("Original real 7×7 / Real original", "original"),
        ("Duplicated feature / Variable duplicada", "singular"),
    ],
    value="original",
    description="Matrix / Matriz:",
    style={"description_width": "110px"},
)

def inspect_matrix(choice):
    A = A_real if choice == "original" else A_singular

    rank = np.linalg.matrix_rank(A)
    s = np.linalg.svd(A, compute_uv=False)
    cond = np.linalg.cond(A)
    full_rank = rank == A.shape[1]

    Ap = np.linalg.pinv(A)
    reconstruction = np.linalg.norm(A @ Ap @ A - A)

    print("Shape / Forma:", A.shape)
    print("Rank / Rango:", rank)
    print("Full rank / Rango completo:", full_rank)
    print("Smallest singular value / Menor valor singular:", f"{s[-1]:.3e}")
    print("Condition number / Número de condición:", f"{cond:.3e}")
    print("Pseudoinverse shape / Forma A+:", Ap.shape)
    print("||A A+ A - A|| =", f"{reconstruction:.3e}")
    print()

    if full_rank:
        print("EN: this square matrix has an ordinary inverse.")
        print("ES: esta matriz cuadrada tiene una inversa ordinaria.")
    else:
        print("EN: rank deficiency proves the ordinary mathematical inverse does not exist.")
        print("ES: la deficiencia de rango demuestra que la inversa matemática ordinaria no existe.")

matrix_output = widgets.interactive_output(
    inspect_matrix,
    {"choice": matrix_choice},
)

display(widgets.VBox([matrix_choice, matrix_output]))

### What are the Moore–Penrose conditions doing? / ¿Qué hacen las condiciones de Moore–Penrose?

You do not need to memorize all four formulas today.

Think of them as consistency rules that make `A⁺` the **unique standard pseudoinverse**.

Two useful ideas are:

- `A A⁺ A ≈ A` → go through the pseudoinverse and return to the original transformation;
- `A⁺ A A⁺ ≈ A⁺` → the pseudoinverse is internally consistent with itself.

The other two conditions make the associated projections symmetric.

> 🇪🇸 No necesitas memorizar las cuatro fórmulas hoy.
>
> Piensa en ellas como reglas de consistencia que hacen de `A⁺` la **pseudoinversa estándar y única**.

## Exercise 2 — one real dataset, three geometries / Ejercicio 2 — un conjunto real, tres geometrías

The full housing design matrix is:

`X.shape = (20433, 7)`

That is a **tall** system.

There is usually no vector of seven coefficients that passes exactly through all 20,433 observations.

So:

`w = X⁺y`

means:

> **Find the seven coefficients that minimize the total squared prediction error.**

This is a **least-squares** solution.

### Residual / Residuo

For one observation:

`residual = actual - predicted`

A residual of zero means the prediction is exact for that observation.

A nonzero residual is not automatically a bug — it is expected when a simple linear model approximates many real observations.

> 🇪🇸 La matriz completa `X` tiene forma `(20433,7)`, por lo que es **alta**.
>
> `w = X⁺y` significa:
>
> **encuentra los siete coeficientes que minimizan el error cuadrático total de predicción.**
>
> El residuo es `real - predicho`. Un residuo distinto de cero es esperable cuando un modelo lineal sencillo aproxima muchas observaciones reales.

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Explain from X.shape why np.linalg.inv(X) is undefined.
# 2. Compute:
#       w_pinv = np.linalg.pinv(X) @ y
# 3. Compare it with:
#       np.linalg.lstsq(X, y, rcond=None)[0]
# 4. Compute predictions and RMSE.
# 5. Explain what a residual means.
#
# ES:
# 1. Explica a partir de X.shape por qué np.linalg.inv(X) no está definida.
# 2. Calcula:
#       w_pinv = np.linalg.pinv(X) @ y
# 3. Compáralo con:
#       np.linalg.lstsq(X, y, rcond=None)[0]
# 4. Calcula predicciones y RMSE.
# 5. Explica qué significa un residuo.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

w_pinv = np.linalg.pinv(X) @ y
w_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]

pred = X @ w_pinv
residual = y - pred
rmse = float(np.sqrt(np.mean(residual**2)))

print("X.shape / Forma de X:", X.shape)
print("EN: X is rectangular, so an ordinary inverse is undefined.")
print("ES: X es rectangular, por lo que una inversa ordinaria no está definida.")
print()
print(
    "pinv and lstsq agree / pinv y lstsq coinciden:",
    np.allclose(w_pinv, w_lstsq),
)
print(f"RMSE / RECM: ${rmse:,.0f}")
print()

for name, coef in zip(column_names, w_pinv):
    print(f"{name:22s} {coef:12.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))

axes[0].scatter(y, pred, s=8, alpha=0.25)
lo = min(y.min(), pred.min())
hi = max(y.max(), pred.max())
axes[0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0].set_xlabel("Actual value / Valor real")
axes[0].set_ylabel("Predicted value / Valor predicho")
axes[0].set_title("Actual vs predicted / Real vs predicho")

axes[1].hist(residual, bins=50)
axes[1].set_xlabel("Actual - predicted / Real - predicho")
axes[1].set_title("Residuals / Residuos")

plt.tight_layout()
plt.show()

print()
print("EN: the pseudoinverse gives the least-squares fit, not a perfect line through all observations.")
print("ES: la pseudoinversa entrega el ajuste de mínimos cuadrados, no una línea perfecta que pase por todas las observaciones.")

### Interactive housing-geometry explorer / Explorador interactivo de geometría de vivienda

Keep the same seven columns and change how many **real rows** are used.

Try:

- `5` rows → wide;
- `7` rows → square;
- `20` or more rows → tall.

The plot shows actual versus predicted values for the selected real observations.

> 🇪🇸 Mantén las mismas siete columnas y cambia cuántas **filas reales** se usan.
>
> Prueba:
>
> - `5` filas → ancho;
> - `7` filas → cuadrado;
> - `20` o más filas → alto.
>
> La gráfica muestra valores reales frente a predichos para las observaciones seleccionadas.

In [ ]:
housing_rows = widgets.IntSlider(
    value=20,
    min=3,
    max=120,
    step=1,
    description="Rows / Filas:",
    continuous_update=False,
    style={"description_width": "105px"},
)

def explore_housing_rows(n_rows):
    A = X[:n_rows]
    b = y[:n_rows]

    w_local = np.linalg.pinv(A) @ b
    pred_local = A @ w_local
    residual_local = b - pred_local

    rank = np.linalg.matrix_rank(A)
    rmse_local = float(np.sqrt(np.mean(residual_local**2)))
    coef_norm = float(np.linalg.norm(w_local))

    if n_rows < A.shape[1]:
        geometry = "wide / ancha"
    elif n_rows == A.shape[1]:
        geometry = "square / cuadrada"
    else:
        geometry = "tall / alta"

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.2, 4))

    ax.scatter(b, pred_local, s=28, alpha=0.7)

    lo = min(b.min(), pred_local.min())
    hi = max(b.max(), pred_local.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--")

    ax.set_xlabel("Actual / Real")
    ax.set_ylabel("Predicted / Predicho")
    ax.set_title(
        f"{geometry} · shape {A.shape} · rank {rank}"
    )

    plt.tight_layout()
    plt.show()

    print("Geometry / Geometría:", geometry)
    print("Shape / Forma:", A.shape)
    print("Rank / Rango:", rank)
    print(f"RMSE / RECM: ${rmse_local:,.0f}")
    print(f"Coefficient norm / Norma de coeficientes: {coef_norm:.3f}")

    if n_rows < A.shape[1]:
        print("EN: fewer equations than unknowns; pinv selects a minimum-norm solution.")
        print("ES: menos ecuaciones que incógnitas; pinv selecciona una solución de norma mínima.")
    elif n_rows == A.shape[1]:
        print("EN: square does not automatically mean invertible; rank still matters.")
        print("ES: cuadrada no significa automáticamente invertible; el rango sigue importando.")
    else:
        print("EN: more equations than unknowns; pinv gives a least-squares fit.")
        print("ES: más ecuaciones que incógnitas; pinv entrega un ajuste de mínimos cuadrados.")

housing_output = widgets.interactive_output(
    explore_housing_rows,
    {"n_rows": housing_rows},
)

display(widgets.VBox([housing_rows, housing_output]))

### Residual explorer / Explorador de residuos

Choose one real California district.

The notebook will show:

- actual median house value;
- predicted value;
- residual = actual − predicted.

> 🇪🇸 Elige un distrito real de California.
>
> El cuaderno mostrará el valor mediano real, el valor predicho y el residuo `real − predicho`.

In [ ]:
district_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=min(500, len(housing) - 1),
    step=1,
    description="District / Distrito:",
    continuous_update=False,
    style={"description_width": "115px"},
)

def inspect_residual(i):
    actual = float(y[i])
    predicted = float(pred[i])
    r = actual - predicted

    print("District index / Índice:", i)
    print(f"Actual / Real:       ${actual:,.0f}")
    print(f"Predicted / Predicho:${predicted:,.0f}")
    print(f"Residual / Residuo:  ${r:,.0f}")
    print()

    if r > 0:
        print("EN: the model under-predicted this observation.")
        print("ES: el modelo subestimó esta observación.")
    elif r < 0:
        print("EN: the model over-predicted this observation.")
        print("ES: el modelo sobreestimó esta observación.")
    else:
        print("EN: exact prediction for this observation.")
        print("ES: predicción exacta para esta observación.")

district_output = widgets.interactive_output(
    inspect_residual,
    {"i": district_slider},
)

display(widgets.VBox([district_slider, district_output]))

## Exercise 3 — pseudoinverse after unfolding real digit images / Ejercicio 3 — pseudoinversa después de desplegar dígitos reales

Take the first 20 real digit images:

`T.shape = (20, 8, 8)`

Unfold them:

`M.shape = (20, 64)`

Read that as:

**20 equations × 64 unknown pixel weights**

This is a **wide** system.

### A target with a known exact solution / Un objetivo con una solución exacta conocida

For every digit, define `b` as its **mean pixel intensity**.

The uniform weight vector:

`[1/64, 1/64, ..., 1/64]`

is one exact solution, because averaging 64 pixels is exactly a linear weighted sum.

But 20 equations cannot uniquely determine 64 unknown weights.

So many exact solutions can exist.

The pseudoinverse chooses the exact solution with the **smallest Euclidean norm**.

> 🇪🇸 Tomamos 20 imágenes reales `8×8` y las desplegamos a una matriz `(20,64)`.
>
> Esto significa **20 ecuaciones y 64 pesos de píxel desconocidos**, por lo que el sistema es ancho.
>
> El vector uniforme `1/64` es una solución exacta conocida para calcular la intensidad media.
>
> Como existen muchas soluciones exactas, la pseudoinversa elige la de **menor norma euclidiana**.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. T = digit_tensor[:20]
# 2. M = unfold(T, 0)               # expected (20, 64)
# 3. b = T.mean(axis=(1, 2))
# 4. x_pinv = np.linalg.pinv(M) @ b
# 5. x_uniform = np.full(64, 1 / 64)
# 6. Verify:
#       M @ x_pinv == b
#       M @ x_uniform == b
# 7. Compare:
#       ||x_pinv|| and ||x_uniform||
# 8. Fold x_pinv back to an 8×8 weight image.
#
# ES:
# 1. Usa las primeras 20 imágenes reales.
# 2. Despliega el tensor a (20,64).
# 3. Construye b con la intensidad media de cada imagen.
# 4. Calcula la solución con pseudoinversa.
# 5. Construye la solución uniforme 1/64.
# 6. Verifica que ambas reproduzcan b.
# 7. Compara sus normas.
# 8. Vuelve a plegar x_pinv como una imagen de pesos 8×8.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

T = digit_tensor[:20]
M = unfold(T, 0)
b = T.mean(axis=(1, 2))

x_pinv = np.linalg.pinv(M) @ b
x_uniform = np.full(M.shape[1], 1 / M.shape[1])

pinv_residual = np.linalg.norm(M @ x_pinv - b)
uniform_residual = np.linalg.norm(M @ x_uniform - b)

pinv_norm = np.linalg.norm(x_pinv)
uniform_norm = np.linalg.norm(x_uniform)

print("Tensor / Tensor:", T.shape)
print("Unfolded / Desplegado:", M.shape)
print("Rank / Rango:", np.linalg.matrix_rank(M))
print()
print("pinv residual / residuo pinv:", f"{pinv_residual:.3e}")
print("uniform residual / residuo uniforme:", f"{uniform_residual:.3e}")
print()
print("||x_pinv||:", f"{pinv_norm:.6f}")
print("||x_uniform||:", f"{uniform_norm:.6f}")
print(
    "Minimum-norm check / Comprobación de norma mínima:",
    pinv_norm <= uniform_norm + 1e-10,
)

x_image = x_pinv.reshape(8, 8)
uniform_image = x_uniform.reshape(8, 8)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))

im0 = axes[0].imshow(x_image, cmap="coolwarm")
axes[0].set_title("Pseudoinverse weights / Pesos pinv")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(uniform_image, cmap="coolwarm")
axes[1].set_title("Uniform weights / Pesos uniformes")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))

plt.tight_layout()
plt.show()

print()
print("EN: both solve the equations, but pinv chooses a solution with no larger Euclidean norm.")
print("ES: ambas soluciones cumplen las ecuaciones, pero pinv elige una solución con norma euclidiana no mayor.")

### Interactive minimum-norm map / Mapa interactivo de norma mínima

Change how many real digit equations are used.

Keep the number below `64` so the system remains wide.

Watch how:

- the rank changes;
- the pseudoinverse weight map changes;
- the solution norm changes.

> 🇪🇸 Cambia cuántas ecuaciones de dígitos reales se usan.
>
> Mantén el número por debajo de `64` para que el sistema siga siendo ancho.
>
> Observa cómo cambian el rango, el mapa de pesos de la pseudoinversa y la norma de la solución.

In [ ]:
digits_slider = widgets.IntSlider(
    value=20,
    min=5,
    max=60,
    step=1,
    description="Digits / Dígitos:",
    continuous_update=False,
    style={"description_width": "115px"},
)

def explore_tensor_pinv(n_digits):
    Tn = digit_tensor[:n_digits]
    Mn = unfold(Tn, 0)
    bn = Tn.mean(axis=(1, 2))

    x = np.linalg.pinv(Mn) @ bn
    uniform = np.full(64, 1 / 64)

    residual_x = np.linalg.norm(Mn @ x - bn)
    residual_uniform = np.linalg.norm(Mn @ uniform - bn)

    rank = np.linalg.matrix_rank(Mn)

    # More horizontal space + constrained_layout avoids overlapping titles
    # and the tight_layout/colorbar warning seen in Colab.
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(13.5, 4.2),
        constrained_layout=True,
    )

    axes[0].imshow(
        Tn[0],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    axes[0].set_title(
        "One real digit\nUn dígito real",
        fontsize=11,
        pad=12,
    )

    vmax = max(
        np.max(np.abs(x)),
        np.max(np.abs(uniform)),
    )

    im1 = axes[1].imshow(
        x.reshape(8, 8),
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[1].set_title(
        "Pseudoinverse minimum-norm\n"
        "Pseudoinversa · norma mínima",
        fontsize=11,
        pad=12,
    )

    im2 = axes[2].imshow(
        uniform.reshape(8, 8),
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[2].set_title(
        "Known uniform solution\n"
        "Solución uniforme conocida",
        fontsize=11,
        pad=12,
    )

    for ax in axes:
        ax.axis("off")

    # Give the two heatmaps their own compact colorbars.
    cbar1 = fig.colorbar(
        im1,
        ax=axes[1],
        fraction=0.046,
        pad=0.04,
    )
    cbar1.set_label(
        "Weight / Peso",
        rotation=90,
        labelpad=10,
    )

    cbar2 = fig.colorbar(
        im2,
        ax=axes[2],
        fraction=0.046,
        pad=0.04,
    )
    cbar2.set_label(
        "Weight / Peso",
        rotation=90,
        labelpad=10,
    )

    fig.suptitle(
        f"Wide system / Sistema ancho — {n_digits} equations × 64 unknowns",
        fontsize=12,
    )

    plt.show()

    print("M.shape / Forma de M:", Mn.shape)
    print("Geometry / Geometría: wide / ancha")
    print("Rank / Rango:", rank)
    print("Unknown weights / Pesos desconocidos:", Mn.shape[1])
    print(f"pinv residual / residuo: {residual_x:.3e}")
    print(f"uniform residual / residuo uniforme: {residual_uniform:.3e}")
    print(f"||x_pinv||: {np.linalg.norm(x):.6f}")
    print(f"||x_uniform||: {np.linalg.norm(uniform):.6f}")
    print()
    print("EN: pinv chooses one particular exact solution from many possibilities.")
    print("ES: pinv elige una solución exacta particular entre muchas posibilidades.")

tensor_output = widgets.interactive_output(
    explore_tensor_pinv,
    {"n_digits": digits_slider},
)

display(widgets.VBox([digits_slider, tensor_output]))

## Quick decision challenge / Reto rápido de decisión

Choose a situation and decide what the correct interpretation is.

> 🇪🇸 Elige una situación y decide cuál es la interpretación correcta.

In [ ]:
decision_choice = widgets.Dropdown(
    options=[
        ("Square + full rank / Cuadrada + rango completo", "inverse"),
        ("Square + rank deficient / Cuadrada + rango deficiente", "singular"),
        ("Tall system / Sistema alto", "tall"),
        ("Wide system / Sistema ancho", "wide"),
    ],
    value="tall",
    description="Case / Caso:",
    style={"description_width": "95px"},
)

def explain_decision(case):
    answers = {
        "inverse": (
            "Ordinary inverse can exist.",
            "Puede existir una inversa ordinaria.",
        ),
        "singular": (
            "No ordinary inverse; use the pseudoinverse and diagnose rank deficiency.",
            "No existe inversa ordinaria; usa la pseudoinversa y diagnostica la deficiencia de rango.",
        ),
        "tall": (
            "Pseudoinverse gives a least-squares solution.",
            "La pseudoinversa entrega una solución de mínimos cuadrados.",
        ),
        "wide": (
            "Pseudoinverse selects the minimum-norm solution among compatible solutions.",
            "La pseudoinversa selecciona la solución de norma mínima entre las soluciones compatibles.",
        ),
    }

    en, es = answers[case]

    print("EN:", en)
    print("ES:", es)

decision_output = widgets.interactive_output(
    explain_decision,
    {"case": decision_choice},
)

display(widgets.VBox([decision_choice, decision_output]))

## What just happened / Qué acaba de pasar

You used the pseudoinverse for **three different reasons**, all anchored in real observations.

### 1. Singular square matrix / Matriz cuadrada singular

Duplicating a real feature made two columns dependent.

The correct mathematical diagnosis was:

`rank < 7`

not simply whether `np.linalg.inv` happened to raise an exception.

The ordinary inverse does not exist, but `A⁺` does.

### 2. Real California housing regression / Regresión real de vivienda

`X.shape = (20433, 7)`

This is a tall system: many more equations than unknown coefficients.

`X⁺y` gives the **least-squares** solution.

### 3. Real digit-image tensor / Tensor real de dígitos

`(20,8,8) → (20,64)`

This is a wide system: 20 equations and 64 unknown weights.

Many exact solutions exist, and the pseudoinverse chooses the **minimum-norm** one.

### The sentence to remember / La frase para recordar

> **Inverse asks for an exact reversible square map. Pseudoinverse gives the standard best-defined solution when that ideal situation is unavailable.**

> 🇪🇸
>
> **La inversa exige un mapa cuadrado, reversible y exacto. La pseudoinversa entrega la solución estándar mejor definida cuando esa situación ideal no está disponible.**

### Geometry summary / Resumen por geometría

- **Tall / Alta** → least squares / mínimos cuadrados
- **Wide / Ancha** → minimum norm / norma mínima
- **Singular / Singular** → ordinary inverse unavailable, pseudoinverse still exists / inversa ordinaria no disponible, pseudoinversa sí existe

### Final self-check / Autoevaluación final

If you see:

`A.shape = (1000, 12)`

what should you ask first?

Not:

**“How do I invert A?”**

Ask:

**“This is tall. What problem am I solving, and does least squares match my goal?”**

> 🇪🇸 Si ves `A.shape = (1000,12)`, no preguntes primero “¿cómo invierto A?”.
>
> Pregunta:
>
> **“Esta matriz es alta. ¿Qué problema estoy resolviendo y mínimos cuadrados corresponde a mi objetivo?”**

---

## Time for Kahoot 🎯 / Hora de Kahoot 🎯

**Kahoot 2 — Einsum, Distance & the Pseudoinverse / Einsum, distancia y la pseudoinversa**  
6 questions / 6 preguntas · about 5 minutes / unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

> 🇪🇸 Entra a **kahoot.it** con el PIN que aparece en la pantalla del facilitador.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-2)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_2_distance_pseudoinverse.xlsx)

Next / Siguiente: **08 · Recursion with matrices and vectors / Recursión con matrices y vectores** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)